# 04 — Maintenance Inspection Priority Index

This notebook calculates a transparent 0—100 decision-support score and selects the top 5% of tree assets for the simulated Year One inspaction program.

The model uses: 

- Simulated conditions
- Clean/imputed DBH
- Simulated inspection recency
- Proximity to schools, sidewalks, trails, and streets
- Equity-priority geography

> The result is an inspection-priority index, not a tree-failure prediction.

In [ ]:
from pathlib import Path
import os
from datetime import datetime
import math
import json
import pandas as pd
import arcpy

arcpy.env.overwriteOutput = True

PROJECT_ROOT = Path(os.environ.get("SEATTLE_TREE_PROJECT_ROOT", Path.cwd())).resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
GDB = PROJECT_ROOT / "data" / "Seattle_Tree_Assets.gdb"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TREES_SPATIAL = str(GDB / "trees_spatial")
TREES_PRIORITY = str(GDB / "trees_priority")
YEAR1_TREES = str(GDB / "inspection_program_year1")

if not arcpy.Exists(TREES_SPATIAL):
    raise FileNotFoundError(
        f"Run revised Notebook 3 first. Missing: {TREES_SPATIAL}"
    )
if arcpy.Exists(TREES_PRIORITY):
    arcpy.management.Delete(TREES_PRIORITY)

arcpy.management.CopyFeatures(TREES_SPATIAL, TREES_PRIORITY)

REFERENCE_DATE = datetime(2026, 1, 1)
ANNUAL_SHARE = 0.05

WEIGHTS = {
    "condition": 0.30,
    "dbh": 0.18,
    "maturity": 0.10,
    "inspection_overdue": 0.15,
    "school": 0.08,
    "sidewalk": 0.07,
    "trail": 0.04,
    "arterial_street": 0.05,
    "equity": 0.03,
}

if abs(sum(WEIGHTS.values()) - 1.0) > 1e-9:
    raise ValueError(f"Weights must sum to 1. Current total: {sum(WEIGHTS.values())}")

pd.DataFrame([
    {"criterion": key,
    "weight": value,
    "weight_pct": value * 100,}
    for key, value in WEIGHTS.items()
])


In [ ]:
def add_field_if_missing(feature_class, field_name, field_type, length=None, alias=None,):
    existing = {field.name for field in arcpy.ListFields(feature_class)}

    if field_name not in existing:
        kwargs = {}
        if length:
            kwargs["field_length"] = length
        if alias:
            kwargs["field_alias"] = alias
        arcpy.management.AddField(feature_class, field_name, field_type, **kwargs,)

fields_to_add = [
    ("OVERDUE_SC", "DOUBLE", None, "Inspection Overdue Score"),
    ("PRIORITY", "DOUBLE", None, "Inspection Priority Index"),
    ("PRIOR_CLASS", "TEXT", 20, "Priority Class"),
    ("ANNUAL_SEL", "SHORT", None, "Selected for Year 1"),
    ("PLAN_YEAR", "SHORT", None, "Recommended Inspection Year"),
    ("INSP_HRS", "DOUBLE", None, "Estimated Inspection Hours"), 
    ("MODEL_VER", "TEXT", 40, "Model Version"),
    ("MODEL_NOTE", "TEXT", 500, "Model Notes"),
]

for args in fields_to_add:
    add_field_if_missing(TREES_PRIORITY, *args)

In [ ]:
required_model_fields = [
    "COND_SCORE", "DBH_SCORE",
    "MATURITY_SC", "LAST_INSP",
    "NEAR_SCHOOL", "NEAR_SIDEWALK",
    "NEAR_TRAIL", "NEAR_ARTERIAL",
    "EQUITY_HIGH",]

available = {f.name for f in arcpy.ListFields(TREES_PRIORITY)}
missing = [field 
           for field in required_model_fields 
           if field not in available]

if missing:
    raise RuntimeError(f"Required model fields are missing: {missing}")
else: 
    print("All fields are accounted for. Go forth and map!")

In [ ]:
cursor_fields = ["COND_SCORE", "DBH_SCORE", "MATURITY_SC", "LAST_INSP", "NEAR_SCHOOL", "NEAR_SIDEWALK", "NEAR_TRAIL",
    "NEAR_ARTERIAL", "EQUITY_HIGH", "OVERDUE_SC", "PRIORITY", "PRIOR_CLASS", "ANNUAL_SEL", "PLAN_YEAR","INSP_HRS",
    "MODEL_VER","MODEL_NOTE",]

with arcpy.da.UpdateCursor(TREES_PRIORITY, cursor_fields,) as rows:
    for row in rows:
        (condition, dbh, maturity, last_insp, school, sidewalk, trail, arterial, equity,) = row[:9]
        years_since = (max(0.0, (REFERENCE_DATE - last_insp).days / 365.25,) if last_insp else 10.0)
        overdue_score = min(years_since / 7.0 * 100.0, 100.0,)

        score = (
            float(condition or 0) * WEIGHTS["condition"]
            + float(dbh or 0) * WEIGHTS["dbh"]
            + float(maturity or 0) * WEIGHTS["maturity"]
            + overdue_score * WEIGHTS["inspection_overdue"]
            + float(school or 0) * 100 * WEIGHTS["school"]
            + float(sidewalk or 0) * 100 * WEIGHTS["sidewalk"]
            + float(trail or 0) * 100 * WEIGHTS["trail"]
            + float(arterial or 0) * 100 * WEIGHTS["arterial_street"]
            + float(equity or 0) * 100 * WEIGHTS["equity"]
        )

        score = round(min(max(score, 0.0), 100.0),)

        if score >= 75:
            priority_class = "Critical"
        elif score >= 55:
            priority_class = "High"
        elif score >= 35:
            priority_class = "Moderate"
        else:
            priority_class = "Low"

        inspection_hours = round(
            0.35
            + 0.003 * float(dbh or 0)
            + 0.002 * float(condition or 0),
            2,
        )

        model_note = ("Priority index uses simulated condition and inspection history, DBH-based maturity, and spatial exposure variables.")
        row[9:] = [overdue_score, score, priority_class, 0, None, inspection_hours, "SeattleTreePriority_v2.0", model_note,]
        rows.updateRow(row)
        
print("Priority scores calculated.")

In [ ]:
oid_field = arcpy.Describe(TREES_PRIORITY).OIDFieldName

rank_rows = [(oid, score or 0) for oid, score in arcpy.da.SearchCursor(TREES_PRIORITY, [oid_field, "PRIORITY"],)]
rank_rows.sort(key=lambda item: (-item[1], item[0]))

selection_count = max(1, math.ceil(len(rank_rows) * ANNUAL_SHARE),)
selected_oids = {oid for oid, score in rank_rows[:selection_count]}
rank_lookup = {oid:rank
              for rank, (oid, score) in enumerate(
                  rank_rows,
                  start=1,
              )}
cohort_size = selection_count

with arcpy.da.UpdateCursor(TREES_PRIORITY, [oid_field, "ANNUAL_SEL", "PLAN_YEAR"],) as rows:
    for oid, annual_selected, plan_year in rows:
        rank = rank_lookup[oid]
        annual_selected = (1 if oid in selected_oids else 0)
        plan_year = (REFERENCE_DATE.year + (rank -1) // cohort_size)
        rows.updateRow([oid, annual_selected, plan_year])

where_clause = (f"{arcpy.AddFieldDelimiters(TREES_PRIORITY, 'ANNUAL_SEL')} = 1")

if arcpy.Exists(YEAR1_TREES):
    arcpy.management.Delete(YEAR1_TREES)

arcpy.analysis.Select(TREES_PRIORITY, YEAR1_TREES, where_clause,)

print(f"Total assets:   {len(rank_rows):,}")
print(f"Year 1 selected: {selection_count:,}")
print(f"Actual selected share: \n{selection_count / len(rank_rows):.2%}")

In [ ]:
weight_df = pd.DataFrame([
    {"criterion": key,
    "weight": value,
    "weight_pct": value * 100,}
    for key, value in WEIGHTS.items()
])

weight_df.to_csv(OUTPUT_DIR / "priority_model_weights_v2.csv", index=False,)

with open(OUTPUT_DIR / "priority_model_config_v2.json", "w", encoding="utf-8",) as file:
    json.dump(
        {"model_version": "SeattleTreePriority_v2.0",
        "annual_share": ANNUAL_SHARE,
        "reference_date": REFERENCE_DATE.isoformat(),
        "weights": WEIGHTS,
        "limitations": [
            "Condition and inspection-history fields are simulated.",
            "Maturity is derived from DBH and is not chronological age.",
            "This model prioritizes inspection and does NOT predict tree failure."
        ],},
        file,
        indent=2,
    )

scores = [row[0] for row in arcpy.da.SearchCursor(TREES_PRIORITY, ["PRIORITY"],)]
score_series = pd.Series(scores)

print("=" * 60)
print("Priority Score Summary")
print("=" * 60)

print(score_series.describe())

print("\nPriority Classes")

priority_counts = {}
with arcpy.da.SearchCursor(TREES_PRIORITY, ["PRIOR_CLASS"],) as cursor:
    for(priority,) in cursor:
        priority_counts[priority] = (priority_counts.get(priority,0) +1)

display(pd.DataFrame(priority_counts.items(), 
    columns = ["Priority Class", "Tree Count",],)
        .sort_values("Priority Class"))

print("\nOutputs")
print(TREES_PRIORITY)
print(YEAR1_TREES)

year1_count = int(arcpy.management.GetCount(YEAR1_TREES)[0])

print(f"Year 1 inspections: {year1_count:,}")
print(f"Percent: {year1_count / 20992 * 100:.2f}%")